# Lecture 1: Introduction to Applied Statistics

## Three decisions, three datasets, one question

A hospital gets fined \$500K for "too many" readmissions. An Airbnb host wonders if they're underpricing by \$50/night. A pharmaceutical company must decide whether a \$2 billion drug actually works.

**How do you make these decisions with data?**

Not formulas — *decisions under uncertainty*.

> *"In God we trust; all others must bring data."* — W. Edwards Deming

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['font.size'] = 12

# Load data
DATA_DIR = 'https://github.com/stanford-mse-125/book/raw/main/data'

## What is applied statistics?

**Applied statistics** is the science of making decisions under uncertainty using data. The data is messy, the sample is limited, and the stakes are real. You'll learn to work at the intersection of three disciplines:

- **Probability** (from MS&E 120) — the mathematics of uncertainty
- **Computing** — the tools for wrangling real datasets
- **Domain knowledge** — the context that turns numbers into insight

As John Tukey put it: *"The best thing about being a statistician is that you get to play in everyone's backyard."* The same tools you'll learn here apply to healthcare, housing, sports, and drug development.

## The three acts of this course

The course follows a three-act structure, and the three datasets we just met map onto the acts:

**Act 1: Build Models** (Lectures 1–7) — Explore data, clean it, and build predictive models. We'll use regression, feature engineering, and decision trees on the Airbnb and hospital datasets.

**Act 2: Trust Models** (Lectures 8–12) — Sampling, hypothesis testing, and regression inference. We'll ask: how precise are our estimates? Is the drug effect real? Which coefficients matter?

**Act 3: See Further** (Lectures 13–19) — Classification, PCA, clustering, time series, tree-based methods, and causal inference. We'll move from "what happened" to "why."

Every week, we'll work with real data — with all the mess that entails.

## A first look: hospital readmissions

Let's start with a real dataset. The Centers for Medicare & Medicaid Services (CMS) tracks how often patients are readmitted to hospitals within 30 days of discharge. Hospitals with "too many" readmissions get fined — up to 3% of their Medicare payments, which translates to \$500K to \$1M per year for a large hospital.

In [ ]:
# Load hospital readmissions data
readmissions = pd.read_csv(f'{DATA_DIR}/hospital-readmissions/hrrp_full.csv')
print(f"Shape: {readmissions.shape[0]:,} rows x {readmissions.shape[1]} columns")
readmissions.head(10)

Each row is one hospital-condition pair: a hospital's readmission performance for a specific condition (heart attack, pneumonia, heart failure, etc.). Tables like this — rectangular grids of rows and columns — are the fundamental data structure in data science. Nearly every dataset you'll encounter in this course lives in a table (or *DataFrame*, in pandas terminology). Let's see what conditions are tracked.

In [ ]:
readmissions['Measure Name'].value_counts()

:::{.callout-important}
## Definition: Excess Readmission Ratio (ERR)
The **Excess Readmission Ratio (ERR)** is the ratio of a hospital's predicted readmission number to its expected number, after adjusting for patient risk. Above 1.0 means more readmissions than expected.
:::

In [ ]:
# Distribution of Excess Readmission Ratios
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(readmissions['Excess Readmission Ratio'].dropna(), bins=50, ax=ax,
             edgecolor='white')
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='Expected = 1.0')
ax.axvline(x=1.05, color='orange', linestyle=':', linewidth=2, label='Your hospital = 1.05')
ax.set_xlabel('Excess Readmission Ratio')
ax.set_ylabel('Count')
ax.set_title('Hospital Readmission Performance Across the U.S.')
ax.legend()
plt.tight_layout()
plt.show()

Notice the distribution is centered near 1.0 — hospitals with ratios above 1.0 have more readmissions than expected, and those below have fewer. But there's real spread.

:::{.callout-tip}
## Think About It
Your hospital's ERR is 1.05 (the orange line). Why might your readmissions be so high? What questions would you ask, or what data would you gather, to understand why — and to figure out what you might do to lower them?
:::

## Another dataset: Airbnb pricing

Now a completely different question. You're an Airbnb host in New York City. You want to set your price. Too high and nobody books; too low and you leave money on the table. What's the right price?

In [ ]:
# Load Airbnb data (just a few key columns for now)
airbnb = pd.read_csv(f'{DATA_DIR}/airbnb/listings.csv', low_memory=False,
                     usecols=['name', 'neighbourhood_group_cleansed', 'room_type',
                              'price', 'bedrooms', 'number_of_reviews'])
print(f"{airbnb.shape[0]:,} listings in NYC")
airbnb.head()

Notice the `price` column — it's stored as a string with `$` and commas. This is what real data looks like: messy. Let's clean it up.

In [ ]:
# Clean price column (remove $ and commas, convert to numeric)
airbnb['price'] = airbnb['price'].astype(str).str.replace('[$,]', '', regex=True).astype(float)
airbnb['price'].describe()

Look at the output above: the mean and the max. And there are listings near \$0. Already the data is telling us something: the "average" might not be very meaningful here. Extreme values — \$0 listings that aren't real prices, and sky-high outliers — distort the mean. *Outliers distort averages* — that's a theme we'll return to all quarter.

In [ ]:
# Price distribution — notice the long right tail
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(airbnb['price'].dropna(), bins=100, ax=ax, edgecolor='white')
ax.set_xlabel('Price per night ($)')
ax.set_ylabel('Count')
ax.set_title('NYC Airbnb Price Distribution')
ax.axvline(airbnb['price'].median(), color='orange', linestyle='--', lw=2,
           label=f'Median = ${airbnb["price"].median():.0f}')
ax.axvline(airbnb['price'].mean(), color='red', linestyle='--', lw=2,
           label=f'Mean = ${airbnb["price"].mean():.0f}')
ax.set_xlim(0, 1000)
ax.legend()
plt.tight_layout()
plt.show()

Notice how far the mean (red) is pulled to the right by expensive listings. The median is a better summary here. We'll dive deep into this data in Lecture 2.

:::{.callout-tip}
## Think About It
If you had a 1-bedroom apartment in Manhattan, would you price at the mean? Why or why not? What other information would you want?
:::

## Why quick analyses go wrong

Suppose you need a fast answer. You hand the hospital dataset to an AI assistant — or to an intern, or to a colleague who skims the columns and writes a quick script. The prompt: *"Which hospitals have the worst readmission rates?"*

### What a quick analysis gets right

- **Fast summary statistics**: histograms, means, counts — all in seconds
- **Decent plots**: bar charts of top/bottom hospitals, distributions by condition
- **Code that runs**: syntactically correct pandas and matplotlib

### What a quick analysis misses

- **No skepticism about the data**: the analysis won't ask "why are 15% of the values missing?" It will silently drop those rows.
- **No domain context**: "Too Few to Report" means small and rural hospitals lack enough cases to report — dropping those rows biases the results toward large urban hospitals.
- **Plausible nonsense**: ranking hospitals by raw readmission count instead of the risk-adjusted ERR would penalize hospitals that treat sicker patients. The ranking would *look* reasonable but be *wrong*.

In [ ]:
# Here's what "Too Few to Report" looks like in the data
non_numeric = readmissions[pd.to_numeric(readmissions['Number of Readmissions'],
                                         errors='coerce').isna()]
print(f"Rows with non-numeric readmission counts: {len(non_numeric):,}")
print(f"That's {len(non_numeric)/len(readmissions)*100:.1f}% of the data")
print()
print("What values do they have?")
print(non_numeric['Number of Readmissions'].value_counts())

:::{.callout-important}
## Definition: Missing Data Mechanism
A **missing data mechanism** describes *why* data is absent — not just *that* it's absent. Here, the data isn't missing randomly; a systematic pattern drives the gaps (small hospitals don't have enough cases to report).
:::

A quick analysis would drop these rows without mentioning it. But *dropping them changes the answer* — the results would be biased toward large urban hospitals.

In [ ]:
# Which hospitals have "Too Few to Report"? Let's look at a sample.
cols_to_show = ['Facility Name', 'State', 'Measure Name', 'Number of Readmissions']
available_cols = [c for c in cols_to_show if c in non_numeric.columns]
non_numeric[available_cols].head(8)

In [ ]:
# How many hospitals are affected, by condition?
fig, ax = plt.subplots(figsize=(8, 5))
(non_numeric['Measure Name']
 .value_counts()
 .plot.barh(ax=ax, color='C3', edgecolor='white'))
ax.set_xlabel('Number of hospitals with "Too Few to Report"')
ax.set_title('Missing data is NOT random — some conditions are harder to track')
plt.tight_layout()
plt.show()

**Catching problems like these is what this course teaches.** Whether the analysis comes from an AI, a colleague, or your own first pass, the habit is the same: verify the data before trusting the conclusions.

## The \$2 billion question

The third scenario we opened with — does this drug work? — is perhaps the highest-stakes application of statistics.

In a clinical trial, you randomly assign patients to get the drug or a placebo. You measure outcomes. Then you ask: is the difference between the groups real, or could it be due to chance?

This question is the domain of **hypothesis testing**, which we'll spend several weeks on in Act 2. The logic in brief:

1. Assume the drug does nothing (the "null hypothesis")
2. Ask: if the drug does nothing, how likely is it that we'd see a difference this large?
3. If the answer is "very unlikely" — below a threshold we set in advance, called the **significance level** — we reject the null hypothesis

We won't analyze clinical trial data today. But notice: the same logic applies whether you're testing a drug, comparing hospitals, or checking if an Airbnb price is unusual. The framework is universal — the context changes everything.

:::{.callout-tip}
## Think About It
In the hospital example, what would the null hypothesis be? (Hint: "This hospital's readmission rate is no different from expected.")
:::

## What you'll be able to do by the end

By the end of MSE 125, you'll be able to:

1. **Explore** a dataset and identify problems before they ruin your analysis
2. **Model** relationships in data using regression and classification
3. **Quantify uncertainty** — not just give a number, but say how confident you are
4. **Reason about causation** — not just correlation. Two variables can be strongly associated without one causing the other; distinguishing association from causation is one of the central challenges of data science.
5. **Critically evaluate** statistical claims — whether they come from a news article, a colleague, or an AI tool

Any analysis can be misleading: the data may be biased, the model may be wrong, or the conclusions may not follow from the evidence. By the end of this course, you'll have the tools to tell the difference.

## Coming up next

In **Lecture 2**, we'll do exploratory data analysis (EDA) on the hospital and Airbnb datasets — distributions, missing data, and outliers.

In **Lecture 3**, we'll tackle data munging and put an AI assistant to the test on real data.

And in the final weeks of the course, we'll return to this hospital data and ask a harder question: does a hospital's readmission rate *cause* its penalty, or is something else going on?

## Key Takeaways

- **Applied statistics is about decisions under uncertainty**, not formulas in a vacuum.
- Real data is messy: missing values, outliers, confounding variables. That mess is the point.
- Any analysis — whether from an AI, a colleague, or your own first pass — deserves skepticism. Your job is to verify.
- The course has three acts: Build Models, Trust Models, See Further. Each act builds on the last.
- Every dataset has a story. Learning to read that story — and question it — is the core skill of a statistician.

## Study guide

### Key ideas

- **Applied statistics** is the science of making decisions under uncertainty using data.
- The **Excess Readmission Ratio (ERR)** compares a hospital's readmissions to what's expected given its patient mix. Above 1.0 = more readmissions than expected.
- A **missing data mechanism** describes *why* data is absent, not just *that* it's absent. In the hospital data, "Too Few to Report" means small hospitals lack enough cases — so their data is systematically suppressed, not randomly missing.
- The **null hypothesis** (preview) is the default assumption that nothing interesting is happening (e.g., "this hospital is no different from average"). Formalized in Lectures 9–10.
- Missing data is often a signal, not just a nuisance — *why* data is missing matters as much as *that* it's missing.
- Any quick analysis — from an AI, a script, or a first pass — can produce plausible-looking results that miss critical problems in the data.
- Outliers distort averages — always look at the distribution, not just summary statistics.

### Computational tools

- `pd.read_csv()` — load a CSV file into a DataFrame
- `.head()` — peek at the first few rows
- `.describe()` — summary statistics (mean, std, min, max, quartiles)
- `.value_counts()` — count unique values in a column
- `sns.histplot()` — plot a histogram
- `pd.to_numeric(errors='coerce')` — convert to numeric, turning non-numbers into NaN

### For the quiz

- Know what the Excess Readmission Ratio measures and what a value above 1.0 means.
- Be able to explain why silently dropping "Too Few to Report" rows biases an analysis.
- Understand the three-act structure of the course (Build Models, Trust Models, See Further).